In [ ]:
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
//var db = OpenOrCreateDatabase(@"P:\hpccluster\RisingBubble2D");
var db = OpenOrCreateDatabase(@"P:\hpccluster\RisingBubble3D");

In [ ]:
var sessions = db.Sessions;
sessions

#0: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_Reinit30_BDF3_testcase1*	09/02/2024 11:00:57	dadb0bce...
#1: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_testcase1_withPreEnforcerActive_restart1*	08/28/2024 15:41:52	a929ee94...
#2: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_testcase1*	08/09/2024 11:54:11	cb26cb13...
#3: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_testcase1_restart1*	08/05/2024 09:45:57	9d1f82be...
#4: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_testcase1_setttingsCheck2	08/01/2024 09:27:13	eff0e8c8...
#5: RisingBubble3D	RB3D_16x16x32AMR1_k2_testcase1_setttingsCheck	04/19/2024 16:25:45	19291211...


In [ ]:
var sess = sessions.Pick(2);
sess

RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_testcase1*	08/09/2024 11:54:11	cb26cb13...

In [ ]:
var tStep_fail = sess.Timesteps.Last();
tStep_fail

 { Time-step: 58; Physical time: 0.5700000000000003s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, GravityZ#A, GravityZ#B, VelocityX@Phi, VelocityY@Phi, VelocityZ@Phi, MPIrank, CutCells; Name:  }

In [ ]:
var tStep_OK = sess.Timesteps.Pick(sess.Timesteps.Count()-2);
tStep_OK

 { Time-step: 57; Physical time: 0.5700000000000003s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, GravityZ#A, GravityZ#B, VelocityX@Phi, VelocityY@Phi, VelocityZ@Phi, MPIrank, CutCells; Name:  }

## Check some Quadrature properties

In [ ]:
using BoSSS.Foundation.Quadrature;

In [ ]:
var tStep = tStep_fail;
//var tStep = tStep_OK;

In [ ]:
SinglePhaseField phi = (SinglePhaseField)tStep.GetField("Phi");
GridData grdData = (GridData)phi.GridDat; 
LevelSet levelSet = new LevelSet(phi.Basis, "levelSet");
levelSet.Acc(1.0, phi); 
LevelSetTracker LsTrk = new LevelSetTracker(grdData, XQuadFactoryHelper.MomentFittingVariants.Saye,  1, new string[] {"A", "B"}, levelSet);
LsTrk.UpdateTracker(tStep.PhysicalTime);

In [ ]:
CellMask CCmask = LsTrk.Regions.GetCutCellMask();
CCmask.ItemEnum.Count()

72

In [ ]:
int order = 2 * phi.Basis.Degree + 1;
XQuadSchemeHelper SchemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order).XQuadSchemeHelper;

In [ ]:
SpeciesId spcId_A = LsTrk.GetSpeciesId("A");
SpeciesId spcId_B = LsTrk.GetSpeciesId("B");

In [ ]:
EdgeQuadratureScheme SurfElem_EdgeA = SchemeHelper.Get_SurfaceElement_EdgeQuadScheme(spcId_A, 0);
CellQuadratureScheme SurfElem_VolumeA = SchemeHelper.Get_SurfaceElement_VolumeQuadScheme(spcId_A, 0);
CellQuadratureScheme VolQuadA = SchemeHelper.GetVolumeQuadScheme(spcId_A);

EdgeQuadratureScheme SurfElem_EdgeB = SchemeHelper.Get_SurfaceElement_EdgeQuadScheme(spcId_B, 0);
CellQuadratureScheme SurfElem_VolumeB = SchemeHelper.Get_SurfaceElement_VolumeQuadScheme(spcId_B, 0);
CellQuadratureScheme VolQuadB = SchemeHelper.GetVolumeQuadScheme(spcId_B);

CellQuadratureScheme LSQuad = SchemeHelper.GetLevelSetquadScheme(0, CCmask);

In [ ]:
double ExecuteEdgeQuad(EdgeQuadratureScheme edgeQuad) {
    double edgeResult = 0.0;
    EdgeQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        edgeQuad.Compile(LsTrk.GridDat, order),
        delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {
            EvalResult.SetAll(1.0);
        },
        delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
            for(int i = 0; i < length; i++) {
                edgeResult += ResultsOfIntegration[i, 0];
            }
        }
    ).Execute();
    return edgeResult;
}

In [ ]:
ExecuteEdgeQuad(SurfElem_EdgeA) - ExecuteEdgeQuad(SurfElem_EdgeB)

0

In [ ]:
double ExecuteCellQuad(CellQuadratureScheme cellQuad) {
    double cellResult = 0.0;
    CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        cellQuad.Compile(LsTrk.GridDat, order),
        delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {
            EvalResult.SetAll(1.0);
        },
        delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
            for(int i = 0; i < length; i++) {
                cellResult += ResultsOfIntegration[i, 0];
            }
        }
    ).Execute();
    return cellResult;
}

In [ ]:
ExecuteCellQuad(SurfElem_VolumeA) - ExecuteCellQuad(SurfElem_VolumeB)

0

In [ ]:
ExecuteCellQuad(VolQuadA) - ExecuteCellQuad(VolQuadA)

0

In [ ]:
ExecuteCellQuad(SurfElem_VolumeA) - ExecuteCellQuad(LSQuad)

0

In [ ]:
var metrics = SchemeHelper.XDGSpaceMetrics;
var CCmetrics = metrics.CutCellMetrics;
var CCmetricsNonAgg = SchemeHelper.NonAgglomeratedMetrics;

In [ ]:
double interfaceArea = 0.0;
int cnt = 0;
List<int> interfaceCells = new List<int>();
foreach (double iA in CCmetrics.InterfaceArea[spcId_A].To1DArray()) {
    interfaceArea += iA;
    if(iA > 0)
        interfaceCells.Add(cnt);
    cnt++;
}
interfaceCells.Count

40

In [ ]:
interfaceArea

1.7216986667233054

In [ ]:
ExecuteCellQuad(LSQuad)

1.7216986667233054

In [ ]:
double interfaceResult = 0.0;
CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
    LSQuad.Compile(LsTrk.GridDat, order),
    delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {
        EvalResult.SetAll(1.0);
    },
    delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
        for(int i = 0; i < length; i++) {
            Console.WriteLine($"cell {i0 + i} - Results of Integration: {ResultsOfIntegration[i, 0]}");
            //Console.WriteLine($"cell {i0 + i} - cutcell metrics interface area: {CCmetrics.InterfaceArea[spcId_A][i0 + i]}");
            double error = (CCmetrics.InterfaceArea[spcId_A][i0 + i] - ResultsOfIntegration[i, 0])/ ResultsOfIntegration[i, 0];
            Console.WriteLine($"cell {i0 + i} - deviation from integration: {error}");
            interfaceResult += ResultsOfIntegration[i, 0];
        }
    }
).Execute();
interfaceResult

cell 135 - Results of Integration: 0.02480831775181165
cell 135 - deviation from integration: 0
cell 136 - Results of Integration: 0.052579589861932396
cell 136 - deviation from integration: 0
cell 137 - Results of Integration: 0.05157247561937746
cell 137 - deviation from integration: 0
cell 138 - Results of Integration: 0.05744451301363209
cell 138 - deviation from integration: 0
cell 139 - Results of Integration: 0.009794351378683414
cell 139 - deviation from integration: 0
cell 175 - Results of Integration: 0.0524011946068431
cell 175 - deviation from integration: 0
cell 179 - Results of Integration: 0.0531668990524527
cell 179 - deviation from integration: 0
cell 180 - Results of Integration: 0.025838676947091365
cell 180 - deviation from integration: 0
cell 215 - Results of Integration: 0.05035626206583527
cell 215 - deviation from integration: 0
cell 220 - Results of Integration: 0.04577074355831945
cell 220 - deviation from integration: 0
cell 221 - Results of Integration: 0.02

1.7216986667233054

In [ ]:
CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
    SchemeHelper.GetVolumeQuadScheme(spcId_A).Compile(LsTrk.GridDat, order),
    delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {
        EvalResult.SetAll(1.0);
    },
    delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
        for(int i = 0; i < length; i++) {
            Console.WriteLine($"cell {i0 + i} - Results of Integration: {ResultsOfIntegration[i, 0]}");
            //Console.WriteLine($"cell {i0 + i} - cutcell metrics interface area: {CCmetrics.InterfaceArea[spcId_A][i0 + i]}");
            double error = (CCmetrics.CutCellVolumes[spcId_A][i0 + i] - ResultsOfIntegration[i, 0])/ ResultsOfIntegration[i, 0];
            Console.WriteLine($"cell {i0 + i} - deviation from integration: {error}");
        }
    }
).Execute();

cell 135 - Results of Integration: 0.00014665897001958197
cell 135 - deviation from integration: 0
cell 136 - Results of Integration: 0.0018885141121614665
cell 136 - deviation from integration: 0
cell 137 - Results of Integration: 0.0020596343370426506
cell 137 - deviation from integration: 0
cell 138 - Results of Integration: 0.0010404801083707274
cell 138 - deviation from integration: 0
cell 139 - Results of Integration: 2.3464220755334075E-05
cell 139 - deviation from integration: 0
cell 175 - Results of Integration: 0.001110420593286974
cell 175 - deviation from integration: 0
cell 176 - Results of Integration: 0.002499999999999999
cell 176 - deviation from integration: 0
cell 177 - Results of Integration: 0.0024999999999999935
cell 177 - deviation from integration: 0
cell 178 - Results of Integration: 0.002499999999999999
cell 178 - deviation from integration: 0
cell 179 - Results of Integration: 0.0018491434783118575
cell 179 - deviation from integration: 0
cell 180 - Results of

## Checking Gauss

In [ ]:
int jCell = LsTrk.Regions.GetCutCellMask().ItemEnum.First();
jCell

135

In [ ]:
BitArray cellIntDomBA = new BitArray(grdData.Cells.NoOfLocalUpdatedCells);
cellIntDomBA[jCell] = true;
CellMask cellIntDom = new CellMask(grdData, cellIntDomBA);

In [ ]:
double result_Interface = 0.0;
CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
    SchemeHelper.GetLevelSetquadScheme(0, cellIntDom).Compile(LsTrk.GridDat, order),
    delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {
        EvalResult.SetAll(1.0);
    },
    delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
        for(int i = 0; i < length; i++) {
            result_Interface += ResultsOfIntegration[i, 0];
        }
    }
).Execute();
result_Interface

0.02480831775181165

In [ ]:
BitArray edgeIntDomBA = new BitArray(grdData.iGeomEdges.Count);
var jCell2Edges = grdData.Cells.Cells2Edges[jCell];
foreach (int iE in jCell2Edges) {
    if (iE < 0) 
        edgeIntDomBA[-iE] = true;
    else 
        edgeIntDomBA[iE] = true;
}
EdgeMask edgeIntDom = new EdgeMask(grdData, edgeIntDomBA);
jCell2Edges

index,value
0,-309
1,-387
2,388
3,389


In [ ]:
double result_CutEdges = 0.0;
EdgeQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
    SchemeHelper.GetEdgeQuadScheme(spcId_A, IntegrationDomain: edgeIntDom).Compile(LsTrk.GridDat, order),
    delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {
        EvalResult.SetAll(1.0);
    },
    delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
        for(int i = 0; i < length; i++) {
            Console.WriteLine($"edge {i0 + i}");
            result_CutEdges += ResultsOfIntegration[i, 0];
        }
    }
).Execute();
result_CutEdges

edge 387
edge 388
edge 389


0.08426426305965004

## Evaluate Interface Continuity

In [ ]:
int order = phi.Basis.Degree;
XQuadSchemeHelper SchemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order).XQuadSchemeHelper;
EdgeQuadratureScheme SurfaceElement_Edge = SchemeHelper.Get_SurfaceElement_EdgeQuadScheme(LsTrk.GetSpeciesId("A"), 0);

In [ ]:
//SurfaceElement_Edge.Domain.ItemEnum.Count()

In [ ]:
EdgeMask CutCellBoundaryEdgeMask = LsTrk.Regions.GetCutCellMask().AllEdges().Except(LsTrk.Regions.GetCutCellMask().GetAllInnerEdgesMask());
//CutCellBoundaryEdgeMask.ItemEnum.Count()

In [ ]:
var factory = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order).XQuadFactoryHelper.GetSurfaceElement_BoundaryRuleFactory(0, LsTrk.GridDat.Grid.RefElements[0]);
EdgeQuadratureScheme SurfaceElement_BoundaryEdge = new EdgeQuadratureScheme(factory, CutCellBoundaryEdgeMask);

In [ ]:
double result = 0;
int D = LsTrk.GridDat.SpatialDimension;
EdgeQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
    SurfaceElement_BoundaryEdge.Compile(LsTrk.GridDat, order),
    delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {

        // inner quadrature node
        NodeSet Enode_l = QR.Nodes;
        int trf = LsTrk.GridDat.Edges.Edge2CellTrafoIndex[i0, 0];
        NodeSet Vnode_l = Enode_l.GetVolumeNodeSet(LsTrk.GridDat, trf, false);
        NodeSet Vnode_g = Vnode_l.CloneAs();
        int cell = LsTrk.GridDat.Edges.CellIndices[i0, 0];
        LsTrk.GridDat.TransformLocal2Global(Vnode_l, Vnode_g, cell);
        if (D == 2)
            Console.WriteLine("inner quadrature node: ({0},{1})", Vnode_g[0, 0], Vnode_g[0, 1]);
        else 
            Console.WriteLine("inner quadrature node: ({0},{1},{2})", Vnode_g[0, 0], Vnode_g[0, 1], Vnode_g[0, 2]);

        // for(int i = 0; i < length; i++) {
        //     EvalResult[i, 0, 0] = 1;    
        // }
        EvalResult.SetAll(1.0);

    },
    delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
        for(int i = 0; i < length; i++) {
            result += ResultsOfIntegration[i, 0];
        }
    }
).Execute();
result

0

In [ ]:
CellQuadratureScheme ContactLineVolumeScheme = SchemeHelper.GetContactLineQuadScheme(LsTrk.GetSpeciesId("A"), 0);

In [ ]:
ContactLineVolumeScheme.Domain.ItemEnum.Count()

0

## Continuity projection

In [ ]:
using BoSSS.Solution.LevelSetTools;

In [ ]:
SinglePhaseField phiDG = (SinglePhaseField)tStep.GetField("PhiDG");

In [ ]:
ContinuityProjection preEnforcer = new ContinuityProjection(
    phi.Basis,
    phiDG.Basis,
    LsTrk.GridDat,
    ContinuityProjectionOption.ConstrainedDG);